In [0]:
# ============================================================
# UTILITIES MAINTENANCE - SILVER LAYER
# DYNAMIC INITIAL + INCREMENTAL PROCESSING
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    LongType
)
from delta.tables import DeltaTable


# ============================================================
# 1. CONFIGURATION
# ============================================================

STORAGE_ACCOUNT = "utilitiesmaintainence"

BRONZE_BASE_PATH = (
    f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
)

SILVER_BASE_PATH = (
    f"abfss://silver@{STORAGE_ACCOUNT}.dfs.core.windows.net/"
)

MONTHLY_FILES_PATH = (
    BRONZE_BASE_PATH + "monthly_files/"
)

REFERENCE_PATH = (
    BRONZE_BASE_PATH + "reference/"
)

PROCESSED_FILES_PATH = (
    SILVER_BASE_PATH + "processed_files/"
)

PROCESSED_FILES_TABLE = (
    "motordatabricks.utilities_silver.processed_files"
)

MAINTENANCE_TABLE = (
    "motordatabricks.utilities_silver.maintenance"
)


# ============================================================
# 2. EXPECTED MAINTENANCE SCHEMA
# ============================================================

maintenance_types = {

    "work_order_id": "string",
    "asset_id": "string",
    "technician_id": "string",

    "asset_type": "string",
    "manufacturer": "string",
    "asset_model": "string",
    "asset_criticality": "string",

    "location_id": "string",
    "city": "string",
    "region": "string",

    "technician_name": "string",
    "technician_skill": "string",
    "maintenance_team": "string",

    "maintenance_type": "string",
    "maintenance_category": "string",
    "planned_flag": "string",
    "failure_type": "string",
    "failure_severity": "string",
    "priority": "string",
    "work_order_status": "string",

    "created_date": "timestamp",
    "scheduled_date": "timestamp",
    "completed_date": "timestamp",

    "downtime_hours": "double",
    "labor_hours": "double",
    "parts_used": "int",

    "parts_cost": "double",
    "labor_cost": "double",
    "total_cost": "double",

    "resolution_code": "string",
    "source_system": "string",
    "updated_at": "timestamp"
}


maintenance_mandatory = [
    "work_order_id",
    "asset_id",
    "technician_id",
    "location_id"
]


# ============================================================
# 3. CONTROL TABLE SCHEMA
# IMPORTANT:
# Defined BEFORE the file-processing TRY block
# ============================================================

success_schema = StructType([
    StructField("file_name", StringType(), True),
    StructField("file_path", StringType(), True),
    StructField("status", StringType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("record_count", LongType(), True)
])


# ============================================================
# 4. GENERIC SOURCE FILE READER
# ============================================================

def read_source_file(
    file_path,
    expected_types,
    mandatory_columns
):

    print("\nReading:", file_path)

    # --------------------------------------------------------
    # Read everything as STRING first
    # --------------------------------------------------------

    raw_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(file_path)
    )

    actual_columns = raw_df.columns


    # --------------------------------------------------------
    # Mandatory column check
    # --------------------------------------------------------

    missing_mandatory = [
        c
        for c in mandatory_columns
        if c not in actual_columns
    ]

    if missing_mandatory:

        raise Exception(
            "Missing mandatory columns: "
            + str(missing_mandatory)
        )


    # --------------------------------------------------------
    # Detect extra columns
    # --------------------------------------------------------

    extra_columns = [
        c
        for c in actual_columns
        if c not in expected_types
    ]

    if extra_columns:

        print(
            "Extra columns detected and ignored:",
            extra_columns
        )


    df = raw_df


    # --------------------------------------------------------
    # Add missing optional columns
    # --------------------------------------------------------

    for column_name, data_type in expected_types.items():

        if column_name not in df.columns:

            print(
                f"Optional column missing: "
                f"{column_name} -> NULL"
            )

            df = df.withColumn(
                column_name,
                F.lit(None).cast(data_type)
            )


    # --------------------------------------------------------
    # Select expected columns only
    # --------------------------------------------------------

    df = df.select(
        list(expected_types.keys())
    )


    # --------------------------------------------------------
    # Cast expected data types
    # --------------------------------------------------------

    for column_name, data_type in expected_types.items():

        df = df.withColumn(
            column_name,
            F.col(column_name).cast(data_type)
        )


    return df


# ============================================================
# 5. CREATE PROCESSED FILE CONTROL TABLE
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {PROCESSED_FILES_TABLE}
(
    file_name STRING,
    file_path STRING,
    status STRING,
    processed_at TIMESTAMP,
    record_count BIGINT
)
USING DELTA
LOCATION '{PROCESSED_FILES_PATH}'
""")


# ============================================================
# 6. VERIFY SILVER MAINTENANCE TABLE
# ============================================================

if not spark.catalog.tableExists(MAINTENANCE_TABLE):

    raise Exception(
        f"Silver table does not exist: {MAINTENANCE_TABLE}"
    )

maintenance_table = DeltaTable.forName(
    spark,
    MAINTENANCE_TABLE
)

print("\nSilver maintenance table found:")
print(MAINTENANCE_TABLE)


# ============================================================
# 7. DISCOVER BRONZE FILES
# ============================================================

bronze_files = [
    file
    for file in dbutils.fs.ls(MONTHLY_FILES_PATH)
    if file.name.lower().endswith(".csv")
]


print("\n==============================================")
print("BRONZE FILE DISCOVERY")
print("==============================================")

print(
    "Total Bronze files:",
    len(bronze_files)
)

for file in bronze_files:

    print(
        file.name
    )


# ============================================================
# 8. READ PROCESSED FILE CONTROL TABLE
# ============================================================

processed_df = spark.table(
    PROCESSED_FILES_TABLE
)

processed_success_df = (
    processed_df
    .filter(
        F.col("status") == "SUCCESS"
    )
)

processed_file_names = {
    row["file_name"]
    for row in (
        processed_success_df
        .select("file_name")
        .collect()
    )
}


# ============================================================
# 9. CHECK SILVER STATE
# ============================================================

silver_maintenance_count = (
    spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM {MAINTENANCE_TABLE}
    """)
    .collect()[0]["cnt"]
)


print("\n==============================================")
print("SILVER STATUS")
print("==============================================")

print(
    "Silver maintenance records:",
    silver_maintenance_count
)

print(
    "Processed SUCCESS files:",
    len(processed_file_names)
)


# ============================================================
# 10. DYNAMIC BOOTSTRAP
# ============================================================

if (
    silver_maintenance_count > 0
    and len(processed_file_names) == 0
):

    print("\n========== DYNAMIC BOOTSTRAP ==========")

    bootstrap_rows = []

    for file in bronze_files:

        record_count = (
            spark.read
            .option("header", True)
            .option("inferSchema", False)
            .csv(file.path)
            .count()
        )

        bootstrap_rows.append(
            (
                file.name,
                file.path,
                "SUCCESS",
                None,
                record_count
            )
        )

        print(
            file.name,
            "->",
            record_count
        )


    if bootstrap_rows:

        bootstrap_df = spark.createDataFrame(
            bootstrap_rows,
            schema=success_schema
        ).withColumn(
            "processed_at",
            F.current_timestamp()
        )


        (
            DeltaTable
            .forName(
                spark,
                PROCESSED_FILES_TABLE
            )
            .alias("target")
            .merge(
                bootstrap_df.alias("source"),
                "target.file_name = source.file_name"
            )
            .whenNotMatchedInsert(
                values={
                    "file_name": "source.file_name",
                    "file_path": "source.file_path",
                    "status": "source.status",
                    "processed_at": "source.processed_at",
                    "record_count": "source.record_count"
                }
            )
            .execute()
        )


# ============================================================
# 11. REFRESH CONTROL TABLE
# ============================================================

processed_df = spark.table(
    PROCESSED_FILES_TABLE
)

processed_file_names = {
    row["file_name"]
    for row in (
        processed_df
        .filter(
            F.col("status") == "SUCCESS"
        )
        .select("file_name")
        .collect()
    )
}


# ============================================================
# 12. DETERMINE NEW FILES
# ============================================================

new_files = [
    file
    for file in bronze_files
    if file.name not in processed_file_names
]


print("\n==============================================")
print("INCREMENTAL FILE DETECTION")
print("==============================================")

print(
    "New files:",
    len(new_files)
)

for file in new_files:

    print(
        "TO PROCESS:",
        file.name
    )


# ============================================================
# 13. PROCESS EACH NEW FILE
# ============================================================

for file in new_files:

    file_name = file.name
    file_path = file.path

    print("\n")
    print("==============================================")
    print("PROCESSING FILE")
    print("==============================================")

    print(
        "File:",
        file_name
    )


    try:

        # ====================================================
        # READ BRONZE FILE
        # ====================================================

        df = read_source_file(
            file_path,
            maintenance_types,
            maintenance_mandatory
        )


        # ====================================================
        # CLEAN STRING COLUMNS
        # ====================================================

        string_columns = [
            column_name
            for column_name, data_type
            in maintenance_types.items()
            if data_type == "string"
        ]


        for c in string_columns:

            df = df.withColumn(
                c,
                F.when(
                    F.trim(
                        F.col(c)
                    ) == "",
                    F.lit(None)
                ).otherwise(
                    F.trim(
                        F.col(c)
                    )
                )
            )


        # ====================================================
        # STANDARDIZE PLANNED FLAG
        # ====================================================

        df = df.withColumn(
            "planned_flag",

            F.when(
                F.upper(
                    F.col("planned_flag")
                ).isin(
                    "YES",
                    "Y",
                    "TRUE",
                    "1"
                ),
                "Y"
            )

            .when(
                F.upper(
                    F.col("planned_flag")
                ).isin(
                    "NO",
                    "N",
                    "FALSE",
                    "0"
                ),
                "N"
            )

            .otherwise(
                F.upper(
                    F.col("planned_flag")
                )
            )
        )


        # ====================================================
        # STANDARDIZE CATEGORIES
        # ====================================================

        categorical_columns = [

            "maintenance_type",
            "maintenance_category",
            "failure_type",
            "failure_severity",
            "priority",
            "work_order_status",
            "resolution_code",
            "source_system"

        ]


        for c in categorical_columns:

            df = df.withColumn(
                c,
                F.when(
                    F.col(c).isNull(),
                    F.lit(None)
                ).otherwise(
                    F.upper(
                        F.col(c)
                    )
                )
            )


        # ====================================================
        # DQ 1 - MANDATORY NULL CHECK
        # ====================================================

        null_failure_count = 0


        for c in maintenance_mandatory:

            count = (
                df
                .filter(
                    F.col(c).isNull()
                )
                .count()
            )

            null_failure_count += count


        # ====================================================
        # DQ 2 - DUPLICATE CHECK
        # ====================================================

        duplicate_count = (
            df
            .groupBy(
                "work_order_id"
            )
            .count()
            .filter(
                F.col("count") > 1
            )
            .count()
        )


        # ====================================================
        # DEDUPLICATION
        # ====================================================

        maintenance_window = (
            Window
            .partitionBy(
                "work_order_id"
            )
            .orderBy(
                F.col(
                    "updated_at"
                ).desc_nulls_last()
            )
        )


        df = (
            df
            .withColumn(
                "_row_number",
                F.row_number()
                .over(
                    maintenance_window
                )
            )
            .filter(
                F.col("_row_number") == 1
            )
            .drop(
                "_row_number"
            )
        )


        # ====================================================
        # DQ 3 - NON-NEGATIVE CHECK
        # ====================================================

        non_negative_columns = [

            "downtime_hours",
            "labor_hours",
            "parts_used",
            "parts_cost",
            "labor_cost",
            "total_cost"

        ]


        negative_count = 0


        for c in non_negative_columns:

            count = (
                df
                .filter(
                    F.col(c).isNotNull()
                    &
                    (F.col(c) < 0)
                )
                .count()
            )

            negative_count += count


        # ====================================================
        # DQ 4 - DATE CHECK
        # ====================================================

        invalid_date_count = (
            df
            .filter(

                F.col(
                    "created_date"
                ).isNotNull()

                &

                F.col(
                    "completed_date"
                ).isNotNull()

                &

                (
                    F.col(
                        "completed_date"
                    )
                    <
                    F.col(
                        "created_date"
                    )
                )
            )
            .count()
        )


        # ====================================================
        # 14. READ REFERENCE DATA
        # ====================================================

        assets_df = read_source_file(

            REFERENCE_PATH + "assets.csv",

            {
                "asset_id": "string"
            },

            [
                "asset_id"
            ]
        )


        locations_df = read_source_file(

            REFERENCE_PATH + "locations.csv",

            {
                "location_id": "string"
            },

            [
                "location_id"
            ]
        )


        technicians_df = read_source_file(

            REFERENCE_PATH + "technicians.csv",

            {
                "technician_id": "string"
            },

            [
                "technician_id"
            ]
        )


        # ====================================================
        # 15. REFERENTIAL INTEGRITY
        # ====================================================

        assets_keys = (
            assets_df
            .select(
                F.trim(
                    F.col("asset_id")
                ).alias("asset_id")
            )
            .dropDuplicates()
        )


        locations_keys = (
            locations_df
            .select(
                F.trim(
                    F.col("location_id")
                ).alias("location_id")
            )
            .dropDuplicates()
        )


        technicians_keys = (
            technicians_df
            .select(
                F.trim(
                    F.col("technician_id")
                ).alias("technician_id")
            )
            .dropDuplicates()
        )


        invalid_asset_count = (
            df
            .select(
                "asset_id"
            )
            .filter(
                F.col("asset_id").isNotNull()
            )
            .join(
                assets_keys,
                "asset_id",
                "left_anti"
            )
            .count()
        )


        invalid_location_count = (
            df
            .select(
                "location_id"
            )
            .filter(
                F.col("location_id").isNotNull()
            )
            .join(
                locations_keys,
                "location_id",
                "left_anti"
            )
            .count()
        )


        invalid_technician_count = (
            df
            .select(
                "technician_id"
            )
            .filter(
                F.col("technician_id").isNotNull()
            )
            .join(
                technicians_keys,
                "technician_id",
                "left_anti"
            )
            .count()
        )


        # ====================================================
        # 16. DQ RESULT
        # ====================================================

        dq_status = "PASS"


        if null_failure_count > 0:
            dq_status = "FAIL"

        if negative_count > 0:
            dq_status = "FAIL"

        if invalid_date_count > 0:
            dq_status = "FAIL"

        if invalid_asset_count > 0:
            dq_status = "FAIL"

        if invalid_location_count > 0:
            dq_status = "FAIL"

        if invalid_technician_count > 0:
            dq_status = "FAIL"


        print("\n========== DQ RESULT ==========")

        print(
            "Mandatory NULL failures:",
            null_failure_count
        )

        print(
            "Duplicate groups:",
            duplicate_count
        )

        print(
            "Negative value failures:",
            negative_count
        )

        print(
            "Invalid date failures:",
            invalid_date_count
        )

        print(
            "Invalid asset IDs:",
            invalid_asset_count
        )

        print(
            "Invalid location IDs:",
            invalid_location_count
        )

        print(
            "Invalid technician IDs:",
            invalid_technician_count
        )

        print(
            "Overall DQ:",
            dq_status
        )


        # ====================================================
        # 17. STOP BAD FILE
        # ====================================================

        if dq_status == "FAIL":

            raise Exception(
                f"DQ failed for {file_name}. "
                "File will not be merged."
            )


        # ====================================================
        # 18. FINAL SILVER DATAFRAME
        # ====================================================

        silver_maintenance_df = df


        incremental_record_count = (
            silver_maintenance_df.count()
        )


        # ====================================================
        # 19. REFRESH DELTA TABLE OBJECT
        # ====================================================

        maintenance_table = DeltaTable.forName(
            spark,
            MAINTENANCE_TABLE
        )


        # ====================================================
        # 20. SAFE SILVER MERGE
        # ====================================================
        #
        # IMPORTANT:
        # Source may contain columns such as:
        # asset_type, manufacturer, asset_model,
        # asset_criticality, city, region,
        # technician_name, technician_skill,
        # maintenance_team
        #
        # Existing Silver maintenance table may NOT contain
        # those columns.
        #
        # Therefore ONLY columns existing in BOTH source
        # and target are used.
        # ====================================================

        target_columns = (
            maintenance_table
            .toDF()
            .columns
        )

        source_columns = (
            silver_maintenance_df
            .columns
        )


        merge_columns = [

            column

            for column in source_columns

            if column in target_columns
            and column != "work_order_id"

        ]


        # ----------------------------------------------------
        # Safety check
        # ----------------------------------------------------

        if "work_order_id" not in target_columns:

            raise Exception(
                "Silver maintenance table does not contain "
                "'work_order_id'."
            )


        if "work_order_id" not in source_columns:

            raise Exception(
                "Source dataframe does not contain "
                "'work_order_id'."
            )


        print("\n========== SILVER MERGE ==========")

        print(
            "Target columns:",
            target_columns
        )

        print(
            "Source columns:",
            source_columns
        )

        print(
            "Common merge columns:",
            merge_columns
        )


        # ----------------------------------------------------
        # UPDATE MAP
        # ----------------------------------------------------

        update_map = {

            column:
            f"source.{column}"

            for column in merge_columns

        }


        # ----------------------------------------------------
        # INSERT MAP
        # ----------------------------------------------------

        insert_map = {

            "work_order_id":
            "source.work_order_id"

        }


        for column in merge_columns:

            insert_map[column] = (
                f"source.{column}"
            )


        # ----------------------------------------------------
        # MERGE
        # ----------------------------------------------------

        (
            maintenance_table

            .alias("target")

            .merge(

                silver_maintenance_df.alias(
                    "source"
                ),

                """
                target.work_order_id
                =
                source.work_order_id
                """

            )

            .whenMatchedUpdate(

                condition="""

                    source.updated_at
                    >
                    target.updated_at

                    OR

                    target.updated_at IS NULL

                """,

                set=update_map

            )

            .whenNotMatchedInsert(

                values=insert_map

            )

            .execute()
        )


        print(
            "Silver MERGE completed successfully."
        )


        # ====================================================
        # 21. MARK FILE SUCCESS
        # ====================================================

        success_df = (

            spark.createDataFrame(

                [
                    (
                        file_name,
                        file_path,
                        "SUCCESS",
                        None,
                        incremental_record_count
                    )
                ],

                schema=success_schema

            )

            .withColumn(
                "processed_at",
                F.current_timestamp()
            )
        )


        (
            DeltaTable
            .forName(
                spark,
                PROCESSED_FILES_TABLE
            )
            .alias("target")
            .merge(

                success_df.alias("source"),

                """
                target.file_name
                =
                source.file_name
                """

            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )


        print(
            f"\nSUCCESS: {file_name}"
        )

        print(
            "Records processed:",
            incremental_record_count
        )


    # ========================================================
    # FILE FAILED
    # ========================================================

    except Exception as e:

        print(
            f"\nFAILED: {file_name}"
        )

        print(
            str(e)
        )


        # ----------------------------------------------------
        # IMPORTANT:
        # success_schema is defined BEFORE the TRY block,
        # so this cannot produce the previous NameError.
        # ----------------------------------------------------

        failed_df = (

            spark.createDataFrame(

                [
                    (
                        file_name,
                        file_path,
                        "FAILED",
                        None,
                        0
                    )
                ],

                schema=success_schema

            )

            .withColumn(
                "processed_at",
                F.current_timestamp()
            )
        )


        (
            DeltaTable
            .forName(
                spark,
                PROCESSED_FILES_TABLE
            )
            .alias("target")
            .merge(

                failed_df.alias("source"),

                """
                target.file_name
                =
                source.file_name
                """

            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )


        # Continue processing remaining files
        continue


# ============================================================
# 22. FINAL SILVER COUNTS
# ============================================================

print("\n==============================================")
print("FINAL SILVER COUNTS")
print("==============================================")


display(

    spark.sql(f"""

        SELECT
            'assets' AS table_name,
            COUNT(*) AS record_count

        FROM
            motordatabricks.utilities_silver.assets


        UNION ALL


        SELECT
            'locations',
            COUNT(*)

        FROM
            motordatabricks.utilities_silver.locations


        UNION ALL


        SELECT
            'technicians',
            COUNT(*)

        FROM
            motordatabricks.utilities_silver.technicians


        UNION ALL


        SELECT
            'maintenance',
            COUNT(*)

        FROM
            {MAINTENANCE_TABLE}

    """)
)


# ============================================================
# 23. PROCESSED FILE CONTROL TABLE
# ============================================================

print("\n==============================================")
print("PROCESSED FILE CONTROL TABLE")
print("==============================================")


display(

    spark.table(
        PROCESSED_FILES_TABLE
    )
    .orderBy(
        "file_name"
    )

)


# ============================================================
# 24. FINAL SUCCESS MESSAGE
# ============================================================

print("\n==============================================")
print("SILVER JOB COMPLETED")
print("==============================================")

print(
    "Bronze files discovered:",
    len(bronze_files)
)

print(
    "Files processed in this run:",
    len(new_files)
)

print(
    "Silver maintenance records:",
    spark.sql(
        f"SELECT COUNT(*) AS cnt FROM {MAINTENANCE_TABLE}"
    ).collect()[0]["cnt"]
)

print(
    "Silver layer processing finished."
)


BRONZE FILE DISCOVERY
Total Bronze files: 3
maintenance_2026_01.csv
maintenance_2026_02.csv
maintenance_2026_03.csv

SILVER STATUS
Silver maintenance records: 1800
Processed SUCCESS files: 3

========== UPDATING RECORD COUNTS ==========
maintenance_2026_01.csv -> 600
maintenance_2026_02.csv -> 600
maintenance_2026_03.csv -> 600
Record counts updated successfully.

INCREMENTAL FILE DETECTION
New files: 0

No new files found.
Silver is already up to date.

FINAL SILVER COUNTS


table_name,record_count
assets,500
locations,72
technicians,100
maintenance,1800



PROCESSED FILE CONTROL TABLE


file_name,file_path,status,processed_at,record_count
maintenance_2026_01.csv,abfss://bronze@utilitiesmaintainence.dfs.core.windows.net/monthly_files/maintenance_2026_01.csv,SUCCESS,2026-08-28T01:09:39.612Z,600
maintenance_2026_02.csv,abfss://bronze@utilitiesmaintainence.dfs.core.windows.net/monthly_files/maintenance_2026_02.csv,SUCCESS,2026-08-28T01:09:39.612Z,600
maintenance_2026_03.csv,abfss://bronze@utilitiesmaintainence.dfs.core.windows.net/monthly_files/maintenance_2026_03.csv,SUCCESS,2026-08-28T01:09:39.612Z,600
